%md
## Streaming Layer: Simulated Trip Events + Structured Streaming

Simulates real-time NYC taxi trip events and processes them using Spark Structured Streaming with windowed aggregation.

**Flow:**
1. Generate simulated trip events, writing each as an individual JSON file to a "landing" folder in ADLS (mimics a real-time event source).
2. Read the landing folder as a stream using Spark's built-in file source.
3. Apply a 1-minute tumbling window aggregation (trip count, avg fare, total fare) per pickup zone, with a watermark for late data.
4. Write results continuously to a Delta table using a 30-second micro-batch trigger.

%md
### Step 1: Simulate Trip Events

Writes individual JSON files to `streaming/landing/`, one per simulated trip.

In [0]:
import json
import random
import time
import uuid
from datetime import datetime

storage_account = "stnyctaxiraj01"
landing_path = f"/mnt/streaming_landing" if False else None  # not used, direct abfss path below

zones = list(range(1, 264))
payment_types = [1, 2, 3, 4]

def generate_trip_event():
    pickup_zone = random.choice(zones)
    dropoff_zone = random.choice(zones)
    distance = round(random.uniform(0.5, 15.0), 2)
    fare = round(distance * random.uniform(2.5, 4.5), 2)
    
    return {
        "trip_id": f"trip_{int(time.time()*1000)}_{random.randint(1000,9999)}",
        "pickup_datetime": datetime.utcnow().isoformat(),
        "PULocationID": pickup_zone,
        "DOLocationID": dropoff_zone,
        "passenger_count": random.randint(1, 4),
        "trip_distance": distance,
        "fare_amount": fare,
        "payment_type": random.choice(payment_types)
    }

landing_dir = f"abfss://streaming@{storage_account}.dfs.core.windows.net/landing/"
num_events = 60

for i in range(num_events):
    trip_event = generate_trip_event()
    file_name = f"{landing_dir}trip_{uuid.uuid4().hex}.json"
    dbutils.fs.put(file_name, json.dumps(trip_event), overwrite=True)
    if (i+1) % 10 == 0:
        print(f"Written {i+1}/{num_events} events...")
    time.sleep(2)

print("Simulation complete.")

%md
### Step 2: Read Landing Folder as a Stream

Structured Streaming requires an explicit schema (unlike batch reads, which can infer it) since the source is open-ended.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import from_json, col, window, count, avg, sum as _sum, current_timestamp

storage_account = "stnyctaxiraj01"
landing_path = f"abfss://streaming@{storage_account}.dfs.core.windows.net/landing/"

trip_event_schema = StructType([
    StructField("trip_id", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("payment_type", IntegerType(), True)
])

df_stream_raw = spark.readStream \
    .format("json") \
    .schema(trip_event_schema) \
    .load(landing_path)

df_stream_raw.printSchema()

%md
### Step 3: Windowed Aggregation

Groups events into 1-minute tumbling windows per pickup zone, with a 1-minute watermark to handle late-arriving events.

In [0]:
from pyspark.sql.functions import to_timestamp

# Convert the string pickup_datetime into a real timestamp Spark can use for windowing
df_stream_parsed = df_stream_raw.withColumn(
    "event_time", to_timestamp(col("pickup_datetime"))
)

df_windowed = df_stream_parsed \
    .withWatermark("event_time", "1 minute") \
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("PULocationID")
    ) \
    .agg(
        count("*").alias("trip_count"),
        avg("fare_amount").alias("avg_fare"),
        _sum("fare_amount").alias("total_fare")
    )

%md
### Step 4: Write Stream to Delta

Uses `append` output mode (safe with a watermark) and a checkpoint location so the stream can resume correctly if restarted.

In [0]:
output_path = f"abfss://streaming@{storage_account}.dfs.core.windows.net/output/"
checkpoint_path = f"abfss://streaming@{storage_account}.dfs.core.windows.net/checkpoint/"

query = df_windowed.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(processingTime="30 seconds") \
    .start(output_path)

%md
### Step 5: Verify Output and Stop the Stream

In [0]:
df_check = spark.read.format("delta").load(output_path)
df_check.show(20, truncate=False)

In [0]:
query.stop()